In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import requests
print("✅ Барлық кітапханалар дайын!")

✅ Барлық кітапханалар дайын!


In [2]:
import requests
import pandas as pd

# USGS-тен соңғы 10 жылдық earthquake деректері (magnitude 5.0+)
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format": "csv",
    "starttime": "2015-01-01",
    "endtime": "2025-01-01",
    "minmagnitude": 5.0,
    "limit": 5000
}

response = requests.get(url, params=params)

with open("earthquake_data.csv", "wb") as f:
    f.write(response.content)

df = pd.read_csv("earthquake_data.csv")
print(f"✅ Деректер жүктелді: {df.shape[0]} жер сілкінісі, {df.shape[1]} баған")
df.head()

✅ Деректер жүктелді: 5000 жер сілкінісі, 22 баған


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2024-12-31T23:13:20.048Z,-20.1826,-70.6231,35.000,5.0,mwr,37.0,129.0,0.470,0.83,...,2025-03-08T22:39:14.040Z,"49 km W of Puerto, Chile",earthquake,3.29,1.918,0.061,26.0,reviewed,us,us
1,2024-12-31T20:09:41.043Z,-6.6675,150.5771,10.000,5.1,mb,61.0,46.0,2.753,0.74,...,2025-03-08T22:39:14.040Z,"124 km ESE of Kandrian, Papua New Guinea",earthquake,9.93,1.340,0.070,66.0,reviewed,us,us
2,2024-12-31T17:09:39.374Z,-4.0500,151.6351,14.523,5.0,mb,59.0,78.0,0.450,0.97,...,2025-03-08T22:39:14.040Z,"60 km WNW of Rabaul, Papua New Guinea",earthquake,3.77,3.912,0.077,53.0,reviewed,us,us
3,2024-12-30T05:49:02.808Z,-17.6555,168.2183,66.612,5.1,mww,121.0,99.0,2.402,0.47,...,2025-03-08T22:39:13.040Z,"13 km NW of Port-Vila, Vanuatu",earthquake,8.57,6.575,0.065,23.0,reviewed,us,us
4,2024-12-30T05:41:06.678Z,-29.9272,-71.9959,10.000,5.5,mww,127.0,76.0,0.658,0.51,...,2025-03-08T22:39:13.040Z,"63 km W of Coquimbo, Chile",earthquake,3.75,1.829,0.058,29.0,reviewed,us,us


In [3]:
# Деректерді зерттеу
print("📊 Баған атаулары:")
print(df.columns.tolist())
print(f"\n📅 Уақыт аралығы: {df['time'].min()} — {df['time'].max()}")
print(f"\n🌍 Магнитуда:")
print(df['mag'].describe())
print(f"\n❌ Бос мәндер:")
print(df.isnull().sum()[df.isnull().sum() > 0])

📊 Баған атаулары:
['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst', 'gap', 'dmin', 'rms', 'net', 'id', 'updated', 'place', 'type', 'horizontalError', 'depthError', 'magError', 'magNst', 'status', 'locationSource', 'magSource']

📅 Уақыт аралығы: 2022-01-04T17:26:32.226Z — 2024-12-31T23:13:20.048Z

🌍 Магнитуда:
count    5000.000000
mean        5.327030
std         0.396198
min         5.000000
25%         5.000000
50%         5.200000
75%         5.500000
max         7.800000
Name: mag, dtype: float64

❌ Бос мәндер:
nst                616
gap                 21
dmin                23
horizontalError     22
magError            36
magNst              26
dtype: int64


In [4]:
# Деректерді тазарту
df_clean = df[['time', 'latitude', 'longitude', 'depth', 'mag', 'place']].copy()

# Уақытты дұрыс форматқа келтіру
df_clean['time'] = pd.to_datetime(df_clean['time'])
df_clean['year'] = df_clean['time'].dt.year
df_clean['month'] = df_clean['time'].dt.month

# Бос мәндерді жою
df_clean = df_clean.dropna()

print(f"✅ Тазартылған деректер: {df_clean.shape[0]} жазба")
print(df_clean.head())

✅ Тазартылған деректер: 5000 жазба
                              time  latitude  longitude   depth  mag  \
0 2024-12-31 23:13:20.048000+00:00  -20.1826   -70.6231  35.000  5.0   
1 2024-12-31 20:09:41.043000+00:00   -6.6675   150.5771  10.000  5.1   
2 2024-12-31 17:09:39.374000+00:00   -4.0500   151.6351  14.523  5.0   
3 2024-12-30 05:49:02.808000+00:00  -17.6555   168.2183  66.612  5.1   
4 2024-12-30 05:41:06.678000+00:00  -29.9272   -71.9959  10.000  5.5   

                                      place  year  month  
0                  49 km W of Puerto, Chile  2024     12  
1  124 km ESE of Kandrian, Papua New Guinea  2024     12  
2     60 km WNW of Rabaul, Papua New Guinea  2024     12  
3            13 km NW of Port-Vila, Vanuatu  2024     12  
4                63 km W of Coquimbo, Chile  2024     12  


In [5]:
# Earthquake карта — Crisis Radar 🌍
fig = px.scatter_geo(
    df_clean,
    lat='latitude',
    lon='longitude',
    color='mag',
    size='mag',
    hover_name='place',
    hover_data={'mag': True, 'depth': True, 'time': True},
    color_continuous_scale='Reds',
    title='🌍 Crisis Radar — Global Earthquake Map (2022-2024)',
    projection='natural earth'
)

fig.update_layout(
    paper_bgcolor='black',
    geo=dict(bgcolor='#0a0a0a', landcolor='#1a1a2e', oceancolor='#16213e'),
    font=dict(color='white')
)

fig.show()

In [6]:
# Магнитуда бойынша үлестіру
fig2 = px.histogram(
    df_clean,
    x='mag',
    nbins=30,
    color_discrete_sequence=['#ff4444'],
    title='📊 Жер сілкінісі магнитудасының үлестірімі',
    labels={'mag': 'Магнитуда', 'count': 'Саны'}
)

fig2.update_layout(
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white'),
    bargap=0.1
)

fig2.show()

In [7]:
# Жылдар бойынша жер сілкінісі саны
monthly = df_clean.groupby(['year', 'month']).size().reset_index(name='count')
monthly['date'] = pd.to_datetime(monthly[['year', 'month']].assign(day=1))

fig3 = px.line(
    monthly,
    x='date',
    y='count',
    title='📈 Жер сілкінісі саны — айлық тренд (2022-2024)',
    labels={'date': 'Уақыт', 'count': 'Жер сілкінісі саны'},
    color_discrete_sequence=['#ff6b6b']
)

fig3.update_layout(
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white'),
)

fig3.show()

In [8]:
# Тереңдік vs Магнитуда
fig4 = px.scatter(
    df_clean,
    x='depth',
    y='mag',
    color='mag',
    color_continuous_scale='Reds',
    title='🔴 Тереңдік vs Магнитуда',
    labels={'depth': 'Тереңдік (км)', 'mag': 'Магнитуда'},
    hover_name='place'
)

fig4.update_layout(
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white')
)

fig4.show()

In [9]:
# Ең қауіпті аймақтар — Top 10 ел
df_clean['country'] = df_clean['place'].str.split(', ').str[-1]

top_countries = df_clean.groupby('country').agg(
    count=('mag', 'count'),
    avg_mag=('mag', 'mean'),
    max_mag=('mag', 'max')
).sort_values('count', ascending=False).head(10).reset_index()

fig5 = px.bar(
    top_countries,
    x='country',
    y='count',
    color='avg_mag',
    color_continuous_scale='Reds',
    title='🌏 Ең қауіпті 10 аймақ',
    labels={'country': 'Ел', 'count': 'Жер сілкінісі саны', 'avg_mag': 'Орт. магнитуда'}
)

fig5.update_layout(
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white'),
    xaxis_tickangle=-45
)

fig5.show()

In [10]:
# STAR Decomposition — аномалияларды бөліп алу
from statsmodels.tsa.seasonal import STL

# Айлық деректер дайындау
ts = monthly.set_index('date')['count']

# STL decomposition
stl = STL(ts, period=12)
result = stl.fit()

# Визуализация
import plotly.graph_objects as go

fig6 = go.Figure()

fig6.add_trace(go.Scatter(x=ts.index, y=result.trend, name='Тренд', line=dict(color='#ff6b6b')))
fig6.add_trace(go.Scatter(x=ts.index, y=result.seasonal, name='Маусымдылық', line=dict(color='#4ecdc4')))
fig6.add_trace(go.Scatter(x=ts.index, y=result.resid, name='Аномалиялар', line=dict(color='#ffe66d')))

fig6.update_layout(
    title='📊 STAR Decomposition — Тренд, Маусымдылық, Аномалиялар',
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white')
)

fig6.show()

In [11]:
# Аномалияларды анықтау
anomaly_threshold = result.resid.std() * 2

anomalies = monthly[abs(result.resid.values) > anomaly_threshold].copy()

print(f"🚨 Аномалия табылды: {len(anomalies)} ай")
print(anomalies[['date', 'count']])

# Визуализация
fig7 = go.Figure()

fig7.add_trace(go.Scatter(
    x=monthly['date'], y=monthly['count'],
    name='Жалпы', line=dict(color='#4ecdc4')
))

fig7.add_trace(go.Scatter(
    x=anomalies['date'], y=anomalies['count'],
    mode='markers', name='Аномалия',
    marker=dict(color='#ff4444', size=12, symbol='x')
))

fig7.update_layout(
    title='🚨 Анықталған аномалиялар',
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white')
)

fig7.show()

🚨 Аномалия табылды: 1 ай
         date  count
23 2023-12-01    281


In [12]:
# ML модель — Earthquake magnitude prediction
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# Features дайындау
df_ml = df_clean.copy()
df_ml['depth_log'] = np.log1p(df_ml['depth'])
df_ml['lat_abs'] = df_ml['latitude'].abs()

X = df_ml[['latitude', 'longitude', 'depth', 'depth_log', 'lat_abs', 'month']]
y = df_ml['mag']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Модельдер
rf = RandomForestRegressor(n_estimators=100, random_state=42)
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

# Нәтижелер
for name, model in [('Random Forest', rf), ('Gradient Boosting', gb)]:
    pred = model.predict(X_test)
    print(f"🤖 {name}:")
    print(f"   MAE: {mean_absolute_error(y_test, pred):.4f}")
    print(f"   R²:  {r2_score(y_test, pred):.4f}\n")

🤖 Random Forest:
   MAE: 0.2898
   R²:  -0.0054

🤖 Gradient Boosting:
   MAE: 0.2745
   R²:  0.0830



In [13]:
# Confidence Intervals — болжам белгісіздігі
predictions = np.array([tree.predict(X_test) for tree in rf.estimators_])

pred_mean = predictions.mean(axis=0)
pred_std = predictions.std(axis=0)
pred_lower = pred_mean - 1.96 * pred_std
pred_upper = pred_mean + 1.96 * pred_std

# Визуализация
fig8 = go.Figure()

x_axis = list(range(100))

fig8.add_trace(go.Scatter(
    x=x_axis, y=pred_upper[:100],
    fill=None, line=dict(color='rgba(255,100,100,0.3)'),
    name='Жоғарғы шек'
))

fig8.add_trace(go.Scatter(
    x=x_axis, y=pred_lower[:100],
    fill='tonexty', line=dict(color='rgba(255,100,100,0.3)'),
    name='Төменгі шек', fillcolor='rgba(255,100,100,0.15)'
))

fig8.add_trace(go.Scatter(
    x=x_axis, y=pred_mean[:100],
    line=dict(color='#ff6b6b'), name='Болжам'
))

fig8.add_trace(go.Scatter(
    x=x_axis, y=y_test.values[:100],
    line=dict(color='#4ecdc4'), name='Нақты мән'
))

fig8.update_layout(
    title='📊 Болжам + Confidence Intervals',
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white')
)

fig8.show()

c:\Users\Alikhan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
c:\Users\Alikhan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
c:\Users\Alikhan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
c:\Users\Alikhan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
c:\Users\Alikhan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X 

In [14]:
# Қай фактор магнитудаға көбірек әсер етеді?
feature_names = ['latitude', 'longitude', 'depth', 'depth_log', 'lat_abs', 'month']
importances = rf.feature_importances_

fig9 = px.bar(
    x=importances,
    y=feature_names,
    orientation='h',
    color=importances,
    color_continuous_scale='Reds',
    title='🔍 Қай фактор маңызды? — Feature Importance',
    labels={'x': 'Маңыздылық', 'y': 'Фактор'}
)

fig9.update_layout(
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white')
)

fig9.show()

In [15]:
# Модельді сақтау
import pickle

with open('../models/rf_model.pkl', 'wb') as f:
    pickle.dump(rf, f)

with open('../models/gb_model.pkl', 'wb') as f:
    pickle.dump(gb, f)

print("✅ Модельдер сақталды!")

✅ Модельдер сақталды!
